# AIPROD — Générer votre Première Vidéo sur Google Colab Pro

**GPU requis :** A100 40GB (Colab Pro)  
**Durée totale :** ~15 minutes (téléchargement) + ~5 minutes (génération)  
**Coût :** Inclus dans Colab Pro (~10$/mois)  

---

### Ce que fait ce notebook

| Étape | Action | Durée |
|-------|--------|-------|
| 1 | Vérifier GPU A100 | 5 sec |
| 2 | Cloner le repo AIPROD | 30 sec |
| 3 | Installer les dépendances | 2 min |
| 4 | Télécharger le modèle LTX-2 19B (18 GB) | 5-10 min |
| 5 | Télécharger le text encoder Gemma3 (2 GB) | 1-2 min |
| 6 | **Générer une vidéo** | 3-5 min |
| 7 | Télécharger le résultat | 10 sec |

> **Résultat :** Un fichier `.mp4` généré par IA à partir de votre prompt texte.

---

### Prérequis

1. **Google Colab Pro** (~10$/mois) — [colab.research.google.com](https://colab.research.google.com)
2. Sélectionner **Runtime > Change runtime type > A100 GPU**
3. C'est tout !

## Étape 1 — Vérifier le GPU

Exécutez cette cellule pour confirmer que vous avez bien un **A100**.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ Pas de GPU détecté !\n"
        "   → Allez dans Runtime > Change runtime type > GPU (A100)\n"
        "   → Si A100 n'apparaît pas, vérifiez votre abonnement Colab Pro"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
vram_free = torch.cuda.mem_get_info(0)[0] / 1024**3

print(f"✅ GPU détecté : {gpu_name}")
print(f"   VRAM : {vram_total:.0f} GB total, {vram_free:.0f} GB libre")
print(f"   PyTorch : {torch.__version__}")
print(f"   CUDA : {torch.version.cuda}")

if vram_total < 20:
    print()
    print("⚠️  ATTENTION : Vous avez un GPU avec < 20 GB VRAM.")
    print("   Le modèle 19B FP8 nécessite ~20 GB minimum.")
    print("   → Changez pour un A100 : Runtime > Change runtime type > A100")
elif vram_total >= 35:
    print()
    print("🎉 Parfait ! A100 40GB détecté — idéal pour AIPROD.")
else:
    print()
    print("✅ GPU suffisant pour la génération vidéo.")

## Étape 2 — Cloner le repo AIPROD

Clone le code source (~7 MB). Les modèles seront téléchargés séparément.

In [ ]:
import os

REPO_URL = "https://github.com/Blockprod/AIPROD.git"
WORK_DIR = "/content/AIPROD"

# Si repo privé, décommentez et ajoutez votre token GitHub :
# REPO_URL = "https://<VOTRE_TOKEN_GITHUB>@github.com/Blockprod/AIPROD.git"

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull origin main

%cd {WORK_DIR}
print(f"\n✅ Repo AIPROD prêt dans {os.getcwd()}")

## Étape 3 — Installer les dépendances

Installe PyTorch + les packages AIPROD (~2 minutes).

In [ ]:
# Installer PyTorch avec CUDA 12.1
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Dépendances essentielles
%pip install -q safetensors einops transformers huggingface_hub
%pip install -q pillow opencv-python-headless imageio rich pydantic pyyaml tqdm
%pip install -q av  # PyAV pour l'encodage vidéo MP4

# Installer les packages AIPROD (mode éditable)
%pip install -q -e packages/aiprod-core
%pip install -q -e packages/aiprod-pipelines

# Vérifier l'installation
import aiprod_core
import aiprod_pipelines
print(f"\n✅ Installation terminée")
print(f"   aiprod-core : OK")
print(f"   aiprod-pipelines : OK")
print(f"   torch : {__import__('torch').__version__}")

## Étape 4 — Télécharger les modèles

Télécharge directement depuis HuggingFace (~20 GB total).

| Modèle | Taille | Source |
|--------|--------|--------|
| LTX-2 19B FP8 (transformer + VAE) | ~18 GB | `Lightricks/LTX-2` |
| Text encoder Gemma3 1B | ~2 GB | `google/gemma-3-1b-pt` |

> **Temps estimé :** 5-10 minutes sur le réseau Google.
> Les fichiers sont sauvegardés sur le disque Colab (~200 GB disponibles).

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import time

# ═══════════════════════════════════════════════════════════════
# 4a. Modèle principal : LTX-2 19B FP8 (~18 GB)
# Contient le transformer SHDT + Video VAE + Audio decoder
# ═══════════════════════════════════════════════════════════════
LTX2_DIR = Path("models/ltx2_research")
LTX2_DIR.mkdir(parents=True, exist_ok=True)

ckpt_file = LTX2_DIR / "ltx-2-19b-dev-fp8.safetensors"

if ckpt_file.exists():
    print(f"✅ LTX-2 19B déjà présent ({ckpt_file.stat().st_size / 1e9:.1f} GB)")
else:
    print("⬇️  Téléchargement LTX-2 19B FP8 (~18 GB)...")
    print("   Source : Lightricks/LTX-2 (HuggingFace)")
    print("   ⏳ Patience, environ 5-10 minutes...")
    t0 = time.time()
    hf_hub_download(
        repo_id="Lightricks/LTX-2",
        filename="ltx-2-19b-dev-fp8.safetensors",
        local_dir=str(LTX2_DIR),
        local_dir_use_symlinks=False,
    )
    elapsed = time.time() - t0
    print(f"✅ LTX-2 19B téléchargé en {elapsed/60:.1f} min")
    print(f"   Taille : {ckpt_file.stat().st_size / 1e9:.1f} GB")

# ═══════════════════════════════════════════════════════════════
# 4b. Text encoder : Gemma 3 1B (~2 GB)
# Encode les prompts texte en embeddings pour le diffusion model
# ═══════════════════════════════════════════════════════════════
TE_DIR = Path("models/text-encoder")

te_model = TE_DIR / "model.safetensors"

if te_model.exists():
    print(f"\n✅ Text encoder déjà présent ({te_model.stat().st_size / 1e9:.1f} GB)")
else:
    print("\n⬇️  Téléchargement text encoder Gemma 3 1B (~2 GB)...")
    t0 = time.time()
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="google/gemma-3-1b-pt",
        local_dir=str(TE_DIR),
        local_dir_use_symlinks=False,
    )
    elapsed = time.time() - t0
    print(f"✅ Text encoder téléchargé en {elapsed:.0f}s")

# ═══════════════════════════════════════════════════════════════
# Résumé
# ═══════════════════════════════════════════════════════════════
print()
print("═" * 60)
print("📦 MODÈLES PRÊTS")
print("═" * 60)
for p in sorted(LTX2_DIR.glob("*.safetensors")):
    print(f"   {p.name} : {p.stat().st_size / 1e9:.1f} GB")
if te_model.exists():
    print(f"   text-encoder/ : {te_model.stat().st_size / 1e9:.1f} GB")
print()

## Étape 5 — GÉNÉRER VOTRE VIDÉO ! 🎬

**Modifiez le `PROMPT` ci-dessous**, puis exécutez la cellule.

| Paramètre | Valeur par défaut | Description |
|-----------|-------------------|-------------|
| `PROMPT` | *Un drone survolant...* | Votre description de la vidéo |
| `NUM_FRAMES` | 61 | Nombre d'images (~2.5 sec à 24fps) |
| `STEPS` | 40 | Qualité (plus = meilleur mais plus lent) |
| `HEIGHT × WIDTH` | 512 × 768 | Résolution |
| `SEED` | 10 | Graine aléatoire (changez pour varier) |

> **Durée estimée :** 3-5 minutes sur A100 pour 61 frames.

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║  CONFIGUREZ VOTRE VIDÉO ICI                                 ║
# ╚═══════════════════════════════════════════════════════════════╝

PROMPT = "A cinematic drone shot over misty mountains at golden hour, \
rays of sunlight piercing through clouds, 4K quality"

# Paramètres de génération
NUM_FRAMES = 61       # 61 = ~2.5 sec, 121 = ~5 sec (plus lent)
STEPS = 40            # 30 = rapide, 40 = standard, 50 = haute qualité
HEIGHT = 512          # Résolution verticale
WIDTH = 768           # Résolution horizontale
SEED = 10             # Changez pour un résultat différent
FP8 = True            # True = moins de VRAM (recommandé)

# ╔═══════════════════════════════════════════════════════════════╗
# ║  NE PAS MODIFIER EN-DESSOUS (sauf si vous savez ce que      ║
# ║  vous faites)                                                ║
# ╚═══════════════════════════════════════════════════════════════╝

import os
import re
import time
import torch

os.chdir("/content/AIPROD")

from aiprod_pipelines.ti2vid_one_stage import TI2VidOneStagePipeline
from aiprod_pipelines.utils.constants import (
    AUDIO_SAMPLE_RATE,
    DEFAULT_AUDIO_GUIDER_PARAMS,
    DEFAULT_NEGATIVE_PROMPT,
    DEFAULT_VIDEO_GUIDER_PARAMS,
)
from aiprod_pipelines.utils.media_io import encode_video

# Chemins des modèles
CHECKPOINT_PATH = "models/ltx2_research"
TEXT_ENCODER_PATH = "models/text-encoder"

# Créer dossier de sortie
OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Nom de fichier basé sur le prompt
safe_name = re.sub(r"[^a-zA-Z0-9_ ]", "", PROMPT)[:50].strip().replace(" ", "_")
OUTPUT_PATH = f"{OUTPUT_DIR}/{safe_name}.mp4"

print("═" * 60)
print("🎬 AIPROD — Génération Vidéo")
print("═" * 60)
print(f"   Prompt   : {PROMPT[:80]}..." if len(PROMPT) > 80 else f"   Prompt   : {PROMPT}")
print(f"   Résolution : {HEIGHT}×{WIDTH}")
print(f"   Frames   : {NUM_FRAMES} (~{NUM_FRAMES/24:.1f}s à 24fps)")
print(f"   Steps    : {STEPS}")
print(f"   Seed     : {SEED}")
print(f"   FP8      : {FP8}")
print(f"   Output   : {OUTPUT_PATH}")
print()

# ── Charger le Pipeline ────────────────────────────────────────
print("⏳ Chargement du pipeline (2-5 min pour le modèle 19B)...")
t_load = time.time()

pipeline = TI2VidOneStagePipeline(
    checkpoint_path=CHECKPOINT_PATH,
    text_encoder_root=TEXT_ENCODER_PATH,
    loras=(),
    fp8transformer=FP8,
)

print(f"✅ Pipeline chargé en {time.time() - t_load:.0f}s")
print()

# ── Générer ────────────────────────────────────────────────────
print(f"🎨 Génération en cours ({STEPS} étapes de diffusion)...")
t_gen = time.time()

with torch.inference_mode():
    video_result, audio_result = pipeline(
        prompt=PROMPT,
        negative_prompt=DEFAULT_NEGATIVE_PROMPT,
        seed=SEED,
        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        frame_rate=24.0,
        num_inference_steps=STEPS,
        video_guider_params=DEFAULT_VIDEO_GUIDER_PARAMS,
        audio_guider_params=DEFAULT_AUDIO_GUIDER_PARAMS,
        images=[],
    )

gen_time = time.time() - t_gen
print(f"✅ Génération terminée en {gen_time:.0f}s")
print()

# ── Convertir et sauvegarder en MP4 ───────────────────────────
print("💾 Encodage MP4...")

# Le pipeline retourne [B, 3, T, H, W] float dans [-1, 1]
# Convertir en [T, H, W, 3] uint8 pour l'encodeur vidéo
if isinstance(video_result, torch.Tensor):
    video = video_result[0]  # [3, T, H, W]
    video = video.permute(1, 2, 3, 0)  # [T, H, W, 3]
    video = ((video.float() + 1.0) / 2.0 * 255.0).clamp(0, 255).to(torch.uint8)
else:
    # Iterator — collecter les chunks
    chunks = list(video_result)
    video = chunks[0] if len(chunks) == 1 else torch.cat(chunks, dim=0)
    if video.ndim == 5:  # [B, 3, T, H, W]
        video = video[0].permute(1, 2, 3, 0)
        video = ((video.float() + 1.0) / 2.0 * 255.0).clamp(0, 255).to(torch.uint8)

encode_video(
    video=video,
    fps=24,
    audio=audio_result if audio_result is not None else None,
    audio_sample_rate=AUDIO_SAMPLE_RATE if audio_result is not None else None,
    output_path=OUTPUT_PATH,
    video_chunks_number=1,
)

# Libérer la VRAM
del pipeline, video_result, audio_result, video
torch.cuda.empty_cache()

# ── Résultat ──────────────────────────────────────────────────
from pathlib import Path
mp4 = Path(OUTPUT_PATH)
size_mb = mp4.stat().st_size / 1024**2
duration = NUM_FRAMES / 24.0

print()
print("═" * 60)
print("🎉 VIDÉO GÉNÉRÉE AVEC SUCCÈS !")
print("═" * 60)
print(f"   📁 Fichier  : {OUTPUT_PATH}")
print(f"   💾 Taille   : {size_mb:.1f} MB")
print(f"   ⏱️  Durée    : {duration:.1f} secondes")
print(f"   📐 Résolution : {HEIGHT}×{WIDTH}")
print(f"   🕐 Temps total : {gen_time:.0f}s")
print()
print("   Exécutez la cellule suivante pour télécharger le fichier.")

## Étape 6 — Télécharger votre vidéo

**Option A :** Téléchargement direct dans votre navigateur  
**Option B :** Sauvegarder sur Google Drive

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPTION A : Téléchargement direct (navigateur)
# ═══════════════════════════════════════════════════════════════
from google.colab import files  # type: ignore[import-not-found]
files.download(OUTPUT_PATH)
print(f"📥 Téléchargement lancé : {OUTPUT_PATH}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPTION B : Sauvegarder sur Google Drive
# ═══════════════════════════════════════════════════════════════
from google.colab import drive  # type: ignore[import-not-found]
import shutil

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/AIPROD/videos'
os.makedirs(DRIVE_DIR, exist_ok=True)

dst = f"{DRIVE_DIR}/{os.path.basename(OUTPUT_PATH)}"
shutil.copy2(OUTPUT_PATH, dst)
print(f"✅ Vidéo sauvegardée sur Drive : {dst}")

## Étape 7 — Visualiser la vidéo dans Colab

Affiche la vidéo directement dans le notebook.

In [ ]:
from IPython.display import HTML
from base64 import b64encode

with open(OUTPUT_PATH, "rb") as f:
    video_data = f.read()

b64 = b64encode(video_data).decode("ascii")
HTML(f"""
<h3>🎬 Résultat : {os.path.basename(OUTPUT_PATH)}</h3>
<video width="768" controls autoplay loop>
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
<p><b>Prompt :</b> {PROMPT}</p>
""")

---

## Bonus — Générer plusieurs vidéos

Modifiez les prompts et lancez cette cellule pour générer un lot de vidéos.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GÉNÉRATION EN LOT — Modifiez les prompts ci-dessous
# ═══════════════════════════════════════════════════════════════

BATCH_PROMPTS = [
    "A peaceful zen garden with cherry blossoms falling in slow motion",
    "A futuristic city at night with neon lights and flying cars",
    "Ocean waves crashing on a tropical beach at sunset, cinematic",
]

BATCH_STEPS = 40
BATCH_FRAMES = 61
BATCH_HEIGHT = 512
BATCH_WIDTH = 768

# ═══════════════════════════════════════════════════════════════

import os, re, time, torch
os.chdir("/content/AIPROD")

from aiprod_pipelines.ti2vid_one_stage import TI2VidOneStagePipeline
from aiprod_pipelines.utils.constants import (
    AUDIO_SAMPLE_RATE, DEFAULT_AUDIO_GUIDER_PARAMS,
    DEFAULT_NEGATIVE_PROMPT, DEFAULT_VIDEO_GUIDER_PARAMS,
)
from aiprod_pipelines.utils.media_io import encode_video

# Charger le pipeline une seule fois
print("⏳ Chargement du pipeline...")
pipeline = TI2VidOneStagePipeline(
    checkpoint_path="models/ltx2_research",
    text_encoder_root="models/text-encoder",
    loras=(),
    fp8transformer=True,
)
print("✅ Pipeline chargé\n")

for i, prompt in enumerate(BATCH_PROMPTS, 1):
    safe_name = re.sub(r"[^a-zA-Z0-9_ ]", "", prompt)[:50].strip().replace(" ", "_")
    out_path = f"/content/output/{safe_name}.mp4"
    
    print(f"\n{'═'*60}")
    print(f"🎬 Vidéo {i}/{len(BATCH_PROMPTS)} : {prompt[:60]}")
    print(f"{'═'*60}")
    
    t0 = time.time()
    with torch.inference_mode():
        video_result, audio_result = pipeline(
            prompt=prompt,
            negative_prompt=DEFAULT_NEGATIVE_PROMPT,
            seed=10 + i,
            height=BATCH_HEIGHT, width=BATCH_WIDTH,
            num_frames=BATCH_FRAMES,
            frame_rate=24.0,
            num_inference_steps=BATCH_STEPS,
            video_guider_params=DEFAULT_VIDEO_GUIDER_PARAMS,
            audio_guider_params=DEFAULT_AUDIO_GUIDER_PARAMS,
            images=[],
        )
    
    # Convertir et sauvegarder
    if isinstance(video_result, torch.Tensor):
        video = video_result[0].permute(1, 2, 3, 0)
        video = ((video.float() + 1.0) / 2.0 * 255.0).clamp(0, 255).to(torch.uint8)
    else:
        chunks = list(video_result)
        video = chunks[0] if len(chunks) == 1 else torch.cat(chunks, dim=0)
        if video.ndim == 5:
            video = video[0].permute(1, 2, 3, 0)
            video = ((video.float() + 1.0) / 2.0 * 255.0).clamp(0, 255).to(torch.uint8)
    
    encode_video(
        video=video, fps=24,
        audio=audio_result if audio_result is not None else None,
        audio_sample_rate=AUDIO_SAMPLE_RATE if audio_result is not None else None,
        output_path=out_path,
        video_chunks_number=1,
    )
    
    size_mb = os.path.getsize(out_path) / 1024**2
    elapsed = time.time() - t0
    print(f"✅ Sauvegardé : {out_path} ({size_mb:.1f} MB, {elapsed:.0f}s)")
    
    del video_result, audio_result, video
    torch.cuda.empty_cache()

# Libérer le pipeline
del pipeline
torch.cuda.empty_cache()

print(f"\n{'═'*60}")
print(f"🎉 {len(BATCH_PROMPTS)} vidéos générées dans /content/output/")
print(f"{'═'*60}")

# Lister les fichiers
for f in sorted(os.listdir("/content/output")):
    if f.endswith(".mp4"):
        size = os.path.getsize(f"/content/output/{f}") / 1024**2
        print(f"   📹 {f} ({size:.1f} MB)")

---

## Bonus — Génération avec LoRA AIPROD (après entraînement)

Si vous avez entraîné le modèle LoRA SHDT (notebook d'entraînement),  
vous pouvez l'appliquer ici pour des résultats personnalisés.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GÉNÉRATION AVEC LORA AIPROD (optionnel — après entraînement)
# ═══════════════════════════════════════════════════════════════
# Décommentez et adaptez le chemin vers votre fichier LoRA :
#
# from aiprod_core.loader import LoraPathStrengthAndSDOps
#
# LORA_PATH = "/content/drive/MyDrive/AIPROD/trained_models/aiprod-shdt-v1-lora/adapter_model.safetensors"
# LORA_STRENGTH = 0.8  # 0.0 = pas d'effet, 1.0 = effet maximum
#
# pipeline = TI2VidOneStagePipeline(
#     checkpoint_path="models/ltx2_research",
#     text_encoder_root="models/text-encoder",
#     loras=[LoraPathStrengthAndSDOps(path=LORA_PATH, strength=LORA_STRENGTH)],
#     fp8transformer=True,
# )

print("ℹ️ Décommentez le code ci-dessus après avoir entraîné votre LoRA.")
print("   Voir le notebook : AIPROD_Sovereign_Training_Colab.ipynb")

---

## FAQ & Dépannage

### ❌ `CUDA out of memory`
- Assurez-vous d'avoir un **A100** (pas T4 ni V100)
- Réduisez `NUM_FRAMES` à 33 ou `HEIGHT/WIDTH` à 384×512
- Vérifiez que `FP8 = True`

### ❌ `No module named 'aiprod_core'`
- Relancez la cellule 3 (installation des packages)
- Vérifiez que `%cd /content/AIPROD` a été exécuté

### ❌ `FileNotFoundError: ltx-2-19b-dev-fp8.safetensors`
- Relancez la cellule 4 (téléchargement modèles)
- Le téléchargement reprend là où il s'est arrêté

### ❌ Vidéo toute noire / bruit
- Essayez un `SEED` différent (ex: 42, 123, 7)
- Augmentez `STEPS` à 50
- Utilisez un prompt plus descriptif en anglais

### 💡 Astuces pour de bons prompts
- Utilisez l'**anglais** pour de meilleurs résultats
- Soyez **descriptif** : mouvement de caméra, éclairage, style
- Ajoutez des mots clés : `cinematic`, `4K`, `slow motion`, `aerial shot`
- Exemples :
  - `"A slow-motion close-up of rain drops falling on a flower petal"`
  - `"Aerial shot of a mountain lake at sunrise, mist rising, 4K cinematic"`
  - `"A cat sitting on a windowsill watching snowfall, cozy room, warm light"`

### 💰 Coût estimé
- Colab Pro : ~10$/mois = **illimité** (dans les limites GPU)
- ~5 min par vidéo → **~50-100 vidéos par session**
- Pas de frais supplémentaires par vidéo